In [7]:
from pathlib import Path
import json

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import pandas as pd
from tqdm import tqdm
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

In [8]:
BASE_DIR = Path("..").resolve()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RESULTS_DIR = BASE_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
ERROR_DIR = RESULTS_DIR / "errors"

In [9]:
ERROR_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [10]:
class URLDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.urls = df["url"].astype(str).tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = self.urls[idx]
        label = self.labels[idx]

        enc = self.tokenizer(
            url,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long),
        }


def bert_predict(df, tokenizer, model, device, batch_size=64, max_len=64):
    ds = URLDataset(df, tokenizer, max_len=max_len)
    dl = DataLoader(ds, batch_size=batch_size)

    model.eval()
    preds, probs, labels = [], [], []

    with torch.no_grad():
        for batch in tqdm(dl, total=len(dl), desc="BERT eval"):
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            out = model(input_ids, attention_mask=mask)
            logits = out.logits
            prob = torch.softmax(logits, dim=1)[:, 1]

            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            probs.extend(prob.cpu().tolist())
            labels.extend(label.cpu().tolist())

    return preds, probs, labels

In [11]:

test_df = pd.read_csv(PROCESSED_DIR / "urls_test.csv")
len(test_df), test_df.head()


(101739,
                                                  url  label  source
 0                boundandgaggedgirls.com/stream.html      0  github
 1  english.turkcebilgi.com/Jean-Talon+(Montreal+M...      0  github
 2  www.poledancefactory.cl/verification/use/111se...      1  github
 3  belfercenter.ksg.harvard.edu/experts/2167/crai...      0  github
 4            baseball-almanac.com/family/fam1a.shtml      0  github)

In [12]:
from pathlib import Path
import json

bert_root = BASE_DIR / "models" / "bert"

# 1) Versuche, den letzten verwendeten BERT-Run aus bert_results.json zu nehmen
bert_results_path = BASE_DIR / "results" / "metrics" / "bert_results.json"
bert_dir = None

if bert_results_path.exists():
    payload = json.loads(bert_results_path.read_text(encoding="utf-8"))
    meta = payload.get("meta", {}) if isinstance(payload, dict) else {}
    # train_bert.py schreibt typischerweise "bert_dir" oder "bert_model_dir"
    meta_path = meta.get("bert_dir") or meta.get("bert_model_dir")
    if meta_path:
        bert_dir = Path(meta_path)
        # falls relativ, relativ zu BASE_DIR auflösen
        if not bert_dir.is_absolute():
            bert_dir = (BASE_DIR / bert_dir).resolve()

# 2) Fallback: automatisch neuesten run_* Ordner mit Gewichten finden
if bert_dir is None or not bert_dir.exists():
    candidates = sorted(
        [p for p in bert_root.glob("run_*") if p.is_dir()],
        key=lambda p: p.name,
        reverse=True,
    )
    for c in candidates:
        if (c / "pytorch_model.bin").exists() or (c / "model.safetensors").exists():
            bert_dir = c
            break

if bert_dir is None:
    raise FileNotFoundError(
        f"Kein gespeichertes BERT-Modell gefunden unter: {bert_root} "
        "(erwartet run_*/pytorch_model.bin oder run_*/model.safetensors)."
    )

print(f"[+] Loading BERT from: {bert_dir}")

tokenizer = DistilBertTokenizerFast.from_pretrained(bert_dir)
model = DistilBertForSequenceClassification.from_pretrained(bert_dir).to(device)

bert_preds, bert_probs, bert_labels = bert_predict(test_df, tokenizer, model, device)

bert_df = test_df.copy()
bert_df["bert_pred"] = bert_preds
bert_df["bert_proba"] = bert_probs

def classify_error(row):
    if row["label"] == 1 and row["bert_pred"] == 1:
        return "TP"
    if row["label"] == 0 and row["bert_pred"] == 0:
        return "TN"
    if row["label"] == 0 and row["bert_pred"] == 1:
        return "FP"
    if row["label"] == 1 and row["bert_pred"] == 0:
        return "FN"
    return "OTHER"

bert_df["error_type"] = bert_df.apply(classify_error, axis=1)
bert_df["error_type"].value_counts()

[+] Loading BERT from: C:\Users\Moritz\Documents\study\aisec\AbschlussProjekt\models\bert\run_20260117_111333


BERT eval: 100%|██████████| 1590/1590 [47:27<00:00,  1.79s/it]


error_type
TN    77539
TP    20835
FN     2324
FP     1041
Name: count, dtype: int64

In [13]:
bert_df.to_csv(ERROR_DIR / "bert_errors_all.csv", index=False)
bert_df[bert_df["error_type"] == "FP"].to_csv(ERROR_DIR / "bert_false_positives.csv", index=False)
bert_df[bert_df["error_type"] == "FN"].to_csv(ERROR_DIR / "bert_false_negatives.csv", index=False)

ERROR_DIR, [p.name for p in ERROR_DIR.glob("bert_*.csv")]

(WindowsPath('C:/Users/Moritz/Documents/study/aisec/AbschlussProjekt/results/errors'),
 ['bert_errors_all.csv',
  'bert_false_negatives.csv',
  'bert_false_positives.csv'])

In [14]:
train_feat = pd.read_csv(PROCESSED_DIR / "urls_train_features.csv")
test_feat = pd.read_csv(PROCESSED_DIR / "urls_test_features.csv")

N_TRAIN = 20000
N_TEST = 50000

if len(train_feat) > N_TRAIN:
    train_feat = train_feat.sample(n=N_TRAIN, random_state=42).reset_index(drop=True)
if len(test_feat) > N_TEST:
    test_feat = test_feat.sample(n=N_TEST, random_state=42).reset_index(drop=True)

len(train_feat), len(test_feat)

(20000, 50000)

In [15]:
class URLInferDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=64):
        self.urls = df["url"].astype(str).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = self.urls[idx]
        enc = self.tokenizer(
            url,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
        }


def get_bert_features(df, tokenizer, model, device, batch_size=64, max_len=64):
    ds = URLInferDataset(df, tokenizer, max_len=max_len)
    dl = DataLoader(ds, batch_size=batch_size)

    model.eval()
    all_logits, all_probs = [], []

    with torch.no_grad():
        for batch in tqdm(dl, total=len(dl), desc="BERT feats"):
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            out = model(input_ids, attention_mask=mask)
            logits = out.logits
            prob = torch.softmax(logits, dim=1)[:, 1]

            all_logits.extend(logits[:, 1].cpu().tolist())
            all_probs.extend(prob.cpu().tolist())

    return pd.DataFrame({"bert_logit": all_logits, "bert_proba": all_probs})

In [16]:

bert_train_feats = get_bert_features(train_feat, tokenizer, model, device)
bert_test_feats = get_bert_features(test_feat, tokenizer, model, device)

train_hybrid = pd.concat([train_feat.reset_index(drop=True), bert_train_feats], axis=1)
test_hybrid = pd.concat([test_feat.reset_index(drop=True), bert_test_feats], axis=1)

train_hybrid.head()

BERT feats: 100%|██████████| 782/782 [23:09<00:00,  1.78s/it]


,url,label,url_len,num_digits,num_letters,num_slash,num_dot,num_dash,num_at,num_qmark,...,kw_verify,kw_secure,kw_update,kw_account,kw_bank,kw_paypal,kw_confirm,kw_free,bert_logit,bert_proba
0,dwefgbghnhgdnhnjhygggbfvafdvfd.3eeweb.com/drop...,1,71,1,64,3,3,0,0,0,...,0,0,0,0,0,0,0,0,2.284221,0.991355
1,hbyutu.com/images/,1,18,0,15,2,1,0,0,0,...,0,0,0,0,0,0,0,0,1.492256,0.958408
2,wopular.com/wireless-now-available-barts-trans...,0,54,0,47,1,1,5,0,0,...,0,0,0,0,0,0,0,0,-2.815433,0.003594
3,metal-metropolis.com/narnia.htm,0,31,0,27,1,2,1,0,0,...,0,0,0,0,0,0,0,0,-1.795817,0.024843
4,winterroseeq.com/,0,17,0,15,1,1,0,0,0,...,0,0,0,0,0,0,0,0,-1.587814,0.036781


In [17]:
drop_cols = ["url", "label"]
feature_cols = [c for c in train_hybrid.columns if c not in drop_cols]

X_train = train_hybrid[feature_cols]
y_train = train_hybrid["label"]

X_test = test_hybrid[feature_cols]
y_test = test_hybrid["label"]

lgb = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42,
)
lgb.fit(X_train, y_train)

hyb_pred = lgb.predict(X_test)
hyb_proba = lgb.predict_proba(X_test)[:, 1]

metrics_hyb = {
    "accuracy": accuracy_score(y_test, hyb_pred),
    "f1": f1_score(y_test, hyb_pred),
    "roc_auc": roc_auc_score(y_test, hyb_proba),
    "pr_auc": average_precision_score(y_test, hyb_proba),
}
metrics_hyb

[LightGBM] [Info] Number of positive: 4561, number of negative: 15439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001484 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1236
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.228050 -> initscore=-1.219355
[LightGBM] [Info] Start training from score -1.219355


{'accuracy': 0.96688,
 'f1': 0.9267451119171901,
 'roc_auc': 0.9911597406289063,
 'pr_auc': 0.9767652978140308}

In [18]:

hyb_df = test_hybrid.copy()
hyb_df["hyb_pred"] = hyb_pred
hyb_df["hyb_proba"] = hyb_proba

def classify_error_h(row):
    if row["label"] == 1 and row["hyb_pred"] == 1:
        return "TP"
    if row["label"] == 0 and row["hyb_pred"] == 0:
        return "TN"
    if row["label"] == 0 and row["hyb_pred"] == 1:
        return "FP"
    if row["label"] == 1 and row["hyb_pred"] == 0:
        return "FN"
    return "OTHER"

hyb_df["error_type"] = hyb_df.apply(classify_error_h, axis=1)
hyb_df["error_type"].value_counts()

error_type
TN    37869
TP    10475
FN      913
FP      743
Name: count, dtype: int64

In [19]:
hyb_df.to_csv(ERROR_DIR / "hybrid_errors_all.csv", index=False)
hyb_df[hyb_df["error_type"] == "FP"].to_csv(ERROR_DIR / "hybrid_false_positives.csv", index=False)
hyb_df[hyb_df["error_type"] == "FN"].to_csv(ERROR_DIR / "hybrid_false_negatives.csv", index=False)

[p.name for p in ERROR_DIR.glob("hybrid_*.csv")]

['hybrid_errors_all.csv',
 'hybrid_false_negatives.csv',
 'hybrid_false_positives.csv']